# Four Umbrellas Portfolio — Colab Backtest

One-click execution of the 20-year backtest in Google Colab.

**Steps:**
1. Run the "Setup" cell (installs dependencies, clones repo)
2. Upload your CSV data to `/content/four-umbrellas-backtest/data/raw/` using the file-upload widget
3. Run the "Backtest" cell
4. Optionally run the "Monte Carlo" cell

See [data/README.md](../data/README.md) in the repository for where to source each CSV.

**Disclaimer:** This notebook produces educational results only — not investment advice. See the full disclaimer at the top of the accompanying Medium article.

## 1. Setup

In [ ]:
# Clone the repo and install dependencies
!git clone https://github.com/padosoft/four-umbrellas-backtest.git
%cd four-umbrellas-backtest
!pip install -q -r requirements.txt

# Fetch the legally-downloadable data
!python fetch_data.py

## 2. Upload MSCI / CBOE / SG / LBMA CSVs

Use the file-upload widget to upload the CSVs you downloaded from MSCI, CBOE, SG, LBMA, S&P, FTSE. Each file must be named exactly as listed in `data/README.md` and placed in `data/raw/`.

In [ ]:
from google.colab import files
import os

os.makedirs('data/raw', exist_ok=True)
uploaded = files.upload()
for filename in uploaded.keys():
    os.rename(filename, f'data/raw/{filename}')

# Validate
!python -m src.data_loader --validate

## 3. Backtest — main run

In [ ]:
# Customize date range and NAV as desired
!python backtest.py --start 2005-01-01 --end 2024-12-31 --nav 100000

## 4. View outputs

In [ ]:
from IPython.display import Image, display
import pandas as pd

print('Summary statistics:')
display(pd.read_csv('output/summary_statistics.csv').set_index('name').T)

for chart in ['equity_curve.png', 'drawdown.png', 'underwater.png',
              'rolling_sharpe.png', 'rolling_returns.png', 'crisis_zoom.png']:
    display(Image(f'output/{chart}'))

## 5. Monte Carlo simulation

In [ ]:
!python backtest.py --monte-carlo --mc-paths 10000 --mc-years 20 --nav 100000

from IPython.display import Image, display
display(Image('output/monte_carlo_fan.png'))
display(Image('output/monte_carlo_distribution.png'))

import pandas as pd
mc_stats = pd.read_csv('output/monte_carlo_stats.csv')
display(mc_stats.T)

## 6. Customize — sensitivity analysis

Edit `src/portfolio.py` to change weights (e.g., gold 0.18 → 0.25, or DBi 0.05 → 0.08), then re-run to see the impact on CAGR / Max DD / Sharpe.